## Анализ вероятности ухода клиента из банка

In [27]:
import time
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from catboost import CatBoostClassifier

RANDOM_STATE = 42

In [28]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [29]:
X, y = train.drop(columns=['id', 'Churn']), train['Churn'].map({'No': 0, 'Yes': 1})

In [30]:
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')
X = preprocessor.fit_transform(X)
X.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,-0.358885,-0.302342,-0.185604,-0.357076,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,-0.358885,0.854793,0.116964,0.545399,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,-0.358885,0.854793,1.111575,1.421875,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-0.358885,-1.419575,0.123402,-1.029637,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,-0.358885,-1.419575,0.147543,-1.029743,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


### Создание Baseline

In [31]:
class modelsHub:
    def __init__(self):
        self.results = []
        self.models = {}

    def _get_scores(self, model, X):
        if hasattr(model, 'predict_proba'):
            return model.predict_proba(X)[:, 1]
        if hasattr(model, 'decision_function'):
            return model.decision_function(X)
        return None

    def choice_best(self, model_class, X_train, y_train, X_valid, y_valid, model_name, params, general_params=None):
        best = {
            'roc_auc': -np.inf,
            'params': None,
            'model': None
        }

        for i in tqdm(range(len(params)), total=len(params), desc=f'{model_name} tuning'):
            model = model_class(**params[i], **general_params) if general_params else model_class(**params[i])
            row = self.fit_predict(
                model,
                X_train,
                y_train,
                X_valid,
                y_valid,
                model_name=model_name,
                params=params[i]
            )
            if row['roc_auc'] > best['roc_auc']:
                best['roc_auc'] = row['roc_auc']
                best['params'] = params[i]
                best['model'] = model
        
        return best['params'], best['model'], best['roc_auc']

    def fit_predict(self, model, X_train, y_train, X_valid, y_valid, model_name, params=None):
        fit_start = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter() - fit_start

        pred_start = time.perf_counter()
        y_pred = model.predict(X_valid)
        pred_time = time.perf_counter() - pred_start

        y_score = self._get_scores(model, X_valid)

        row = {
            'model': model_name,
            'fit_time_sec': fit_time,
            'predict_time_sec': pred_time,
            'accuracy': accuracy_score(y_valid, y_pred),
            'precision': precision_score(y_valid, y_pred, zero_division=0),
            'recall': recall_score(y_valid, y_pred, zero_division=0),
            'f1': f1_score(y_valid, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_valid, y_score) if y_score is not None else np.nan,
        }

        if params:
            for k, v in params.items():
                row[f'param_{k}'] = v

        self.results.append(row)
        self.models[model_name] = model
        return row

    def leaderboard(self, only_best=False):
        if not self.results:
            return pd.DataFrame()

        df = pd.DataFrame(self.results)

        if only_best:
            def _best_index(g):
                if g['roc_auc'].notna().any():
                    return g['roc_auc'].idxmax()
                return g.index[0]
            best_indices = df.groupby('model', group_keys=False).apply(_best_index).values
            df = df.loc[best_indices].reset_index(drop=True)

        df_sorted = df.sort_values('roc_auc', ascending=False).reset_index(drop=True)

        def highlight_best_roc_auc(row):
            best_roc_auc = df_sorted['roc_auc'].max()
            if pd.notna(row['roc_auc']) and row['roc_auc'] == best_roc_auc:
                return ['background-color: #F0FFF0'] * len(row)
            return [''] * len(row)

        return (
            df_sorted.style
            .format({
                'fit_time_sec': '{:.4f}',
                'predict_time_sec': '{:.4f}',
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
            })
            .apply(highlight_best_roc_auc, axis=1)
        )

In [32]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

hub = modelsHub()
hub.fit_predict(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_train,
    y_train,
    X_valid,
    y_valid,
    model_name='LogisticRegression',
)
hub.leaderboard()

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc
0,LogisticRegression,0.8016,0.0426,0.8563,0.6879,0.6623,0.6749,0.9085


### RandomForest

### CatBoost

In [ ]:
depth = [4, 6, 8]
iterations = [100, 1000, 5000]
reg_lambdas = [0.1, 0.5]

param_grid = list(product(depth, iterations, reg_lambdas))
model, best_params, best_roc_auc = hub.choice_best(
    model_class=CatBoostClassifier,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    model_name='CatBoostClassifier',
    params=[{'depth': d, 'iterations': it, 'reg_lambda': rl} for d, it, rl in param_grid],
    general_params={'random_state': RANDOM_STATE, 'verbose': 0},
)

print(best_params)

CatBoostClassifier tuning:  11%|█         | 2/18 [00:03<00:28,  1.79s/it]

In [ ]:
hub.leaderboard(only_best=True)


,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc
0,CatBoostClassifier,62.1441,0.1269,0.8627,0.7162,0.6469,0.6798,0.9169
1,LogisticRegression,0.5625,0.0111,0.8563,0.6879,0.6623,0.6749,0.9085
